# DIMBA — Shakespeare (Kaggle Dual T4 → HuggingFace)

**~50M params | fp16 mixed | Dual T4 | ~5-10 min | Auto-upload private to HF**

⚡ Before running:
1. Upload `dimba-lib-exp.zip` as Kaggle dataset, add to notebook
2. Add `HF_TOKEN` in Kaggle → Settings → Secrets
3. Set Accelerator to **GPU T4 x2**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import os, sys, torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}')

## 1. Setup

In [ ]:
import os, sys, shutil, urllib.request

INPUT_SRC = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train_shakespeare.py' in files:
        INPUT_SRC = os.path.dirname(root)
        break
if INPUT_SRC is None:
    raise FileNotFoundError('Dataset not found! Add dimba-lib-exp as input.')

print(f'Found: {INPUT_SRC}')
REPO = '/kaggle/working/dimba-lib-exp'
if os.path.exists(REPO):
    shutil.rmtree(REPO)
shutil.copytree(INPUT_SRC, REPO)
os.chdir(REPO)
sys.path.insert(0, 'src')

!pip install -q -e .
!pip install -q tensorboard huggingface_hub

# Mamba SSD CUDA kernels: fast, low-memory selective scan (replaces the
# pure-PyTorch fallback). --no-build-isolation reuses the installed torch at
# build time. First install may take a few minutes if it compiles from source.
!pip install -q --no-build-isolation causal-conv1d mamba-ssm
try:
    import mamba_ssm
    print('OK mamba_ssm', getattr(mamba_ssm, '__version__', '?'))
except Exception as e:
    print('WARNING mamba_ssm import failed:', e)

os.makedirs('data', exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
    'data/shakespeare.txt')
print(f'Repo: {REPO}')
print(f'Data: {os.path.getsize("data/shakespeare.txt"):,} bytes  |  Ready.')

## 2. Train

d_model=512, 8 layers, dual T4, batch=128 — **~50M params**

In [ ]:
!python scripts/train_shakespeare.py \
    --d-model 512 --num-layers 8 \
    --seq-len 128 --num-steps 128 \
    --d-state 64 --use-mamba-ssm \
    --epochs 15 --batch-size 64 \
    --accelerator gpu --devices 2 --strategy auto \
    --precision 16-mixed

## 3. Generate Samples

In [ ]:
import torch
from dimba.training import DIMBALightningModule
from dimba.tokenizers import SimpleCharacterTokenizer
from dimba.diffusion.sampling import sample_from_model

# save_hyperparameters() stored the full model_config in the checkpoint, so this
# rebuilds the EXACT model (same kernel + dims) and loads the weights. No fragile
# shape-introspection, and training/inference always use the same SSM backend.
module = DIMBALightningModule.load_from_checkpoint(
    'checkpoints/shakespeare/shakespeare1.ckpt', map_location='cuda')
model = module.model.cuda().eval()

cfg = dict(module.hparams['model_config'])
n_params   = sum(p.numel() for p in model.parameters())
outer_d    = cfg['d_model']
d_latent   = cfg.get('d_latent', outer_d)
num_layers = cfg['num_denoiser_layers']
print(f'{n_params:,} params | d_model={outer_d} latent={d_latent} layers={num_layers}')
print(f"ssm kernel : {type(model.denoiser.blocks[0].mamba_fwd).__name__}")

tok = SimpleCharacterTokenizer(vocab_size=128)
for p in ['Juliet:', 'Hamlet:', 'To be or not to', 'Now is the winter of our', 'First Citizen:']:
    ids = torch.tensor([tok.encode(p)], device='cuda')
    with torch.no_grad():
        gen = sample_from_model(model, prompt_ids=ids, seq_len=200, num_steps=50, temperature=0.8, top_p=0.95)
    print(f'\n  {p}')
    print(f'  {tok.decode(gen[0])[:200]}')
    print('  ' + '-' * 50)

## 4. Upload to HuggingFace (private)

In [ ]:
from huggingface_hub import HfApi, create_repo
from datetime import datetime

token = os.environ.get('HF_TOKEN')
if not token:
    print('ERROR: HF_TOKEN not set. Kaggle → Settings → Secrets → Add HF_TOKEN')
else:
    api = HfApi(token=token)
    username = api.whoami()['name']
    repo_id = f'{username}/dimba-shakespeare'
    create_repo(repo_id, private=True, token=token, exist_ok=True)

    api.upload_file(
        path_or_fileobj='checkpoints/shakespeare/shakespeare1.ckpt',
        path_in_repo='shakespeare1.ckpt',
        repo_id=repo_id, token=token)
    api.upload_file(
        path_or_fileobj='checkpoints/shakespeare/tokenizer.json',
        path_in_repo='tokenizer.json',
        repo_id=repo_id, token=token)

    card = f'''---
license: mit
tags: [dimba, diffusion, mamba, shakespeare]
---
# DIMBA Shakespeare
Latent-diffusion Mamba trained on tinyshakespeare.
- Params: {n_params:,}
- Config: d_model={outer_d}, latent={d_latent}, layers={num_layers}
- Trained: {datetime.now().strftime("%Y-%m-%d")}
'''
    api.upload_file(
        path_or_fileobj=card.encode(),
        path_in_repo='README.md',
        repo_id=repo_id, token=token)

    print(f'https://huggingface.co/{repo_id} (private)')